In [0]:
SRC = "/Volumes/energy/ops/raw_source"

for f in dbutils.fs.ls(SRC):
    print(f"{f.name:60s} {f.size/1e6:8.2f} MB")

for f in dbutils.fs.ls(SRC):
    if f.name.lower().endswith(".csv"):
        print("\n" + "=" * 80)
        print(f.name)
        print("=" * 80)
        print(dbutils.fs.head(f.path, 600))

In [0]:
%pip install openpyxl

In [0]:
from openpyxl import load_workbook
SRC = "/Volumes/energy/ops/raw_source"

for name in ["September 2021.xlsx", "October 2022.xlsx", "January 2024- June 2025.xlsx"]:
    wb = load_workbook(f"{SRC}/{name}", read_only=True, data_only=True)
    print("=" * 80)
    print(name, "| sheets:", wb.sheetnames)
    for ws in wb.worksheets[:3]:
        print("--- sheet:", ws.title)
        for row in ws.iter_rows(min_row=1, max_row=6, values_only=True):
            print(row[:10])
    wb.close()

In [0]:
from pyspark.sql import functions as F

k = spark.read.option("header", "true").csv(f"{SRC}/long_data_.csv")
d = F.to_date("Dates", "dd/MM/yyyy HH:mm:ss")

display(k.select(
    F.count("*").alias("rows"),
    F.countDistinct("States").alias("states"),
    F.countDistinct("Dates").alias("distinct_dates"),
    F.min(d).alias("first_date"),
    F.max(d).alias("last_date"),
    F.sum(F.when(d.isNull(), 1).otherwise(0)).alias("unparsed_dates"),
))

In [0]:
for name in ["September 2021.xlsx", "October 2022.xlsx"]:
    wb = load_workbook(f"{SRC}/{name}", read_only=True, data_only=True)
    ws = wb["Sheet1"]
    rows = ws.iter_rows(min_row=1, max_row=2, values_only=True)
    print("=" * 80)
    print(name, "| dimensions:", ws.calculate_dimension())
    for r in rows:
        print(r)
    wb.close()

wb = load_workbook(f"{SRC}/January 2024- June 2025.xlsx", read_only=True, data_only=True)
print("=" * 80)
print("Jan 2024 - Jun 2025 | Report dimensions:", wb["Report"].calculate_dimension())
wb.close()

In [0]:
dup = (k.groupBy("States", "Dates").count().where("count > 1")
         .groupBy("Dates").agg(F.count("*").alias("states_duplicated"),
                               F.max("count").alias("max_copies")))
display(dup.orderBy(F.to_date("Dates", "dd/MM/yyyy HH:mm:ss")))

# Do the duplicates have identical Usage values or different ones?
display(k.groupBy("States", "Dates")
         .agg(F.count("*").alias("n"), F.countDistinct("Usage").alias("distinct_usage"))
         .where("n > 1")
         .groupBy("distinct_usage").count())

In [0]:
kk = (k.withColumn("date", F.to_date("Dates", "dd/MM/yyyy HH:mm:ss"))
       .withColumn("usage", F.col("Usage").cast("double")))

display(kk.where("States IN ('Tamil Nadu', 'Maharashtra', 'UP') "
                 "AND date BETWEEN '2019-07-01' AND '2019-07-19'")
          .groupBy("States", "date")
          .agg(F.collect_list("usage").alias("values"))
          .orderBy("States", "date"))

In [0]:
pairs = kk.groupBy("States", "date").count().where("count > 1").select("States", "date")
dups = kk.join(pairs, ["States", "date"])
others = kk.select(F.col("States").alias("s2"),
                   F.col("date").alias("other_date"),
                   F.col("usage").alias("u2"))

display(dups.join(others, (F.col("States") == F.col("s2"))
                          & (F.col("usage") == F.col("u2"))
                          & (F.col("date") != F.col("other_date")))
            .groupBy("date", "other_date").count()
            .orderBy(F.desc("count"))
            .limit(20))

In [0]:
w = spark.read.option("header", "true").csv(f"{SRC}/dataset_tk.csv")
display(w.where(F.col("_c0").like("%/07/2019%"))
         .select("_c0", "Tamil Nadu", "Maharashtra", "UP")
         .orderBy("_c0"))